# Master instruction —Feature Engineering-breed

Je werkt als senior research assistant voor een masterthesis in data analysis.
Voor meer informatie over de thesis / onderzoeksvoorstel / opzet: bekijk bron "Geannoteerd_onderzoeksvoorstel.md" en voor extra gelinkte literatuur bron; "External factors and SHAP in Urban Parking copy.pdf" (beide bestanden zijn te vinden in de map 'literatuur_en_info' (binnen dit project)) 

voor structuur en gewenste flow check projectalomvattede: "README.md"

Context:
- Check hieronder de vooropgestelde feature engineering notebookstructuur:
  1. fe_00_design_and_split_lock.ipynb
  2. fe_01_build_core_features_TSE.ipynb
  3. fe_02_build_forecast_lag_features_L.ipynb
  4. fe_03_finalize_feature_sets_and_export.ipynb
- Projectfase: Phase 3 — Feature Engineering
- Phase 2 (EDA) is afgerond, dit is DE basis voor interne consistentie en keuzes voor FE!!
  - Belangrijk is ook gelijkaardige stijl als phase 2 in phase 3 over te nemen, maar dan specifiek op feature engineering toegepast natuurlijk
- shortterm is de primaire modelbasis
  - (longterm blijft enkel robustness-context en mag niet het hoofddesign sturen)
- Er zijn twee modeltracks (meer uitleg is te vinden in: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/literatuur_en_info/Geannoteerd_onderzoeksvoorstel copy.md):
  1. forecast track: autoregressieve features toegestaan
  2. policy track: occupancy-lags NIET toegestaan
- De operationele split ligt vast:
  - train = 2020, 2023, 2024
  - holdout = 2025
  - 2019 uitgesloten wegens volledig missende weatherfeatures
- Alle transformaties moeten train-only gefit worden
- Alle tijdsfeatures moeten strikt causaal opgebouwd worden
- Model- en featurekeuze gebeurt later via rolling/blocked CV binnen train
- 2025 blijft volledig locked voor finale evaluatie

Doel van phase 3:
- een reproduceerbare, leakage-safe, immutable featureset bouwen
- consistente schema’s afleveren voor forecast en policy
- features expliciet structureren in blokken (cfr. onderzoeksvoorstel):
  - T = time structure
  - S = spatial identity
  - E = external factors
  - L = autoregressive structure (alleen forecast)
- alle featurekeuzes moeten inhoudelijk verdedigbaar zijn vanuit EDA en onderzoeksdoel

Belangrijke werkinstructies:
1. Werk notebook-native en reproduceerbaar.
2. Gebruik enkel train-only fitting voor alle fit-afhankelijke transformaties.
3. Splits train en holdout vroeg en hard; nooit lekken van 2025 naar train.
4. Maak feature engineering expliciet causaal:
   - geen toekomstinformatie
   - geen target-proxies
   - geen smoothing over toekomstige observaties
   - geen niet-causale aggregaties
5. Gebruik duidelijke helperfuncties en schema-validatie.
6. Werk primair op shortterm.
7. longterm mag enkel gebruikt worden voor beperkte robustness-context, niet als primaire pipelinebasis.
8. Maak alle featurebeslissingen expliciet en academisch verdedigbaar.
9. Sluit elk notebook af met:
   - "What was built"
   - "Leakage and validity checks"
   - "Implications for next notebook"
10. Indien iets ambigu is, kies de meest leakage-veilige optie en documenteer die.

Schrijf elke interpretatieve markdown-sectie alsof ze later kan worden herwerkt tot tekst voor de masterthesis.

Stijlregels:
- helder, academisch, voorzichtig
- geen losse bullet dump als lopende tekst beter is
- benoem richting, grootteorde, onzekerheid en beperking
- maak expliciet waarom het resultaat relevant is voor de volgende fase

Jouw taak is niet alleen code schrijven, maar een thesis-waardige feature engineering pipeline ontwerpen.

## Notebooksepcifieke prompt
Maak notebook `fe_01_build_core_features_TSE.ipynb`.

Doel:
De gedeelde kernfeatures bouwen voor beide tracks: T (time), S (spatial), E (external).

Belangrijke context:
- Werk primair op shortterm
- Split train/holdout is al gefixeerd
- Alles wat fitting vereist moet enkel op train gefit worden
- Dit notebook mag GEEN occupancy-lags bouwen
- De output van dit notebook moet bruikbaar zijn voor zowel forecast als policy

Bouw de features in deze volgorde:
... vul aan/ vervang waar nodig !! bekijk ook altijd nog eens goed de bevindingen uit stage 2
A. T — Time structure
1. Maak cyclische encodings voor:
   - hour
   - weekday_int
   - month
2. Maak interpreteerbare kalenderfeatures:
   - weekend indicator
   - holiday indicators
   - school vacation indicators
   - combined day class / day_type variant indien EDA dit ondersteunt
   - 2020 regime indicator
3. Evalueer ordinale versus cyclische tijdsrepresentatie conceptueel en documenteer waarom cyclisch de standaard wordt.
4. Zorg dat alle tijdsfeatures puur observation-based en niet-lekkend zijn.

B. S — Spatial identity
5. Bouw spatial/entity-features die thesis-consistent zijn:
   - parking_location_category / tier-representatie
   - log(total_capacity)
   - eventuele capacity buckets indien EDA dit ondersteunt
   - parking identity encoding op leakage-veilige manier
6. Indien target encoding gebruikt wordt:
   - fit alleen op train
   - gebruik smoothing / shrinkage
   - documenteer exact hoe leakage vermeden wordt
7. Indien behavioral clusters gebruikt worden:
   - fit alleen op train
   - pas nadien toe op holdout
   - documenteer clusterlogica en stabiliteit
8. Voeg enkel spatial features toe die later ook interpreteerbaar blijven.

C. E — External factors
9. Bouw weatherfeatures op basis van EDA-bevindingen:
   - niet-lineaire neerslagrepresentatie
   - temperatuurtransformaties of seizoeninteracties
   - wind threshold feature(s)
   - zon/sunshine feature(s)
   - ... vul aan/ vervang waar nodig !! 
10. Bouw kalender/eventfeatures:
   - event day
   - type-dummies
   - event scale
   - n_concurrent_events
   - ... vul aan/ vervang waar nodig !!  
11. Bouw interacties die in EDA sterk gemotiveerd waren, maar houd ze beheersbaar:
   - event × tier
   - holiday × tier
   - vacation × tier
   - weather × season 
   - ... vul aan/ vervang waar nodig !! confer stage 2
12. Bouw event cascade features zonder leakage:
   - hours_to_event
   - hours_since_event
   - clipped windows
   - sentinel-waarden waar geen event relevant is, en leg ook suuuuuper duidelijk uit en motiveer zeer kritisch!!!!
13. Gebruik geen vrije tekstvelden zoals event_names_combined als ruwe predictor.

D. Validatie
14. Maak systematische checks:
   - missings per feature
   - constant columns
   - cardinaliteit
   - train/holdout schema-equivalentie
   - onverwachte leakage-risico’s
15. Vul de feature registry automatisch aan.

E. Slot
16. Exporteer een gedeelde TSE-featurelaag voor train en holdout.
17. Schrijf een academische slotsectie:
   - welke featurefamilies zijn gebouwd?
   - welke interacties zijn behouden of bewust niet behouden?
   - welke risico’s blijven bestaan?
   - welke stap volgt in forecast-only L features?

Belangrijk:
- Geef voor elke featurefamilie een korte inhoudelijke motivatie vanuit EDA.
- Voeg alleen interacties toe die inhoudelijk verdedigbaar zijn.
- Houd de featurelaag rijk, maar niet nodeloos explosief.

## 0. Setup and contracts

This notebook builds the shared `T + S + E` feature layer for both tracks.

Hard constraints:
- Primary basis: `shortterm` only.
- Locked split: train (`2020, 2023, 2024`) and holdout (`2025`).
- Any fit-dependent transform is fit on train only.
- No occupancy lag or rolling feature is created in this notebook.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Project root not found")


def as_bool_flag(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(0).astype(float).gt(0)


def as_int_flag(series: pd.Series) -> pd.Series:
    return as_bool_flag(series).astype(np.int8)


def cyclical_encode(series: pd.Series, period: int) -> tuple[np.ndarray, np.ndarray]:
    vals = pd.to_numeric(series, errors="coerce").astype(float)
    angle = 2.0 * np.pi * (vals % period) / period
    return np.sin(angle), np.cos(angle)


def slugify(value: str) -> str:
    s = str(value).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"


def ensure_datetime(df: pd.DataFrame, col: str) -> None:
    df[col] = pd.to_datetime(df[col], errors="coerce")


def qcut_with_fallback(series: pd.Series, q: int, labels: list[str]) -> tuple[pd.Series, np.ndarray]:
    ser = pd.to_numeric(series, errors="coerce")
    try:
        binned, edges = pd.qcut(ser, q=q, labels=labels, retbins=True, duplicates="drop")
    except ValueError:
        edges = np.array([ser.min(), ser.max()], dtype=float)
        binned = pd.cut(ser, bins=edges, labels=labels[:1], include_lowest=True)
    return binned, edges


def cut_using_edges(series: pd.Series, edges: np.ndarray, labels: list[str]) -> pd.Series:
    ser = pd.to_numeric(series, errors="coerce")
    edges_adj = np.array(edges, dtype=float)
    edges_adj = np.unique(edges_adj)
    if len(edges_adj) < 2:
        edges_adj = np.array([ser.min(), ser.max() if ser.max() > ser.min() else ser.min() + 1.0])
    if len(labels) != len(edges_adj) - 1:
        labels = labels[: len(edges_adj) - 1]
    return pd.cut(ser, bins=edges_adj, labels=labels, include_lowest=True)


def compute_event_cascade_features(
    timestamp_series: pd.Series,
    event_series: pd.Series,
    clip_hours: int = 72,
) -> pd.DataFrame:
    timeline = (
        pd.DataFrame(
            {
                "rounded_hour": pd.to_datetime(timestamp_series, errors="coerce"),
                "event_flag": as_int_flag(event_series),
            }
        )
        .dropna(subset=["rounded_hour"])
        .groupby("rounded_hour", as_index=False)["event_flag"]
        .max()
        .sort_values("rounded_hour")
        .reset_index(drop=True)
    )

    sentinel = clip_hours + 1

    if timeline.empty or timeline["event_flag"].sum() == 0:
        mapping = pd.DataFrame(
            {
                "rounded_hour": timeline["rounded_hour"],
                "e_hours_to_event_clip": sentinel,
                "e_hours_since_event_clip": sentinel,
                "e_event_proximity_clip": sentinel,
                "e_event_window_pre_24h": 0,
                "e_event_window_post_24h": 0,
            }
        )
        return mapping

    ts = timeline["rounded_hour"].values.astype("datetime64[h]").astype("int64")
    event_ts = timeline.loc[timeline["event_flag"].eq(1), "rounded_hour"].values.astype("datetime64[h]").astype("int64")

    idx_next = np.searchsorted(event_ts, ts, side="left")
    idx_prev = np.searchsorted(event_ts, ts, side="right") - 1

    to_hours = np.full(len(ts), np.inf, dtype=float)
    valid_next = idx_next < len(event_ts)
    to_hours[valid_next] = event_ts[idx_next[valid_next]] - ts[valid_next]

    since_hours = np.full(len(ts), np.inf, dtype=float)
    valid_prev = idx_prev >= 0
    since_hours[valid_prev] = ts[valid_prev] - event_ts[idx_prev[valid_prev]]

    to_clip = np.where(np.isfinite(to_hours), np.minimum(to_hours, clip_hours), sentinel)
    since_clip = np.where(np.isfinite(since_hours), np.minimum(since_hours, clip_hours), sentinel)
    prox_clip = np.minimum(to_clip, since_clip)

    mapping = pd.DataFrame(
        {
            "rounded_hour": timeline["rounded_hour"],
            "e_hours_to_event_clip": to_clip.astype(int),
            "e_hours_since_event_clip": since_clip.astype(int),
            "e_event_proximity_clip": prox_clip.astype(int),
            "e_event_window_pre_24h": ((to_clip <= 24) & (timeline["event_flag"].eq(0))).astype(np.int8),
            "e_event_window_post_24h": ((since_clip <= 24) & (timeline["event_flag"].eq(0))).astype(np.int8),
        }
    )

    return mapping


PROJECT_ROOT = find_project_root()
SPLIT_DIR = PROJECT_ROOT / "data_processed" / "phase3_splits"
OUTPUT_DIR = PROJECT_ROOT / "data_processed" / "phase3_features" / "core_tse"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_YEARS = [2020, 2023, 2024]
HOLDOUT_YEARS = [2025]

KEY_COLS = ["parking_id", "rounded_hour", "year", "occupancy_rate", "operational_split"]

print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)

Project root: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2
Output dir: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed/phase3_features/core_tse


## 1. Load locked split and verify scope

We load the already locked split artifacts from `phase3_splits` and re-check the temporal contract before any feature construction.

In [2]:
train_path = SPLIT_DIR / "MAD_shortterm_train_2020_2023_2024.parquet"
holdout_path = SPLIT_DIR / "MAD_shortterm_holdout_2025.parquet"

train_raw = pd.read_parquet(train_path)
holdout_raw = pd.read_parquet(holdout_path)

for df in [train_raw, holdout_raw]:
    ensure_datetime(df, "rounded_hour")
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

train_years = sorted(train_raw["year"].dropna().astype(int).unique().tolist())
holdout_years = sorted(holdout_raw["year"].dropna().astype(int).unique().tolist())

assert train_years == TRAIN_YEARS, f"Unexpected train years: {train_years}"
assert holdout_years == HOLDOUT_YEARS, f"Unexpected holdout years: {holdout_years}"

scope_check_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_raw),
            "n_parkings": train_raw["parking_id"].nunique(),
            "date_min": train_raw["rounded_hour"].min(),
            "date_max": train_raw["rounded_hour"].max(),
            "years": ", ".join(map(str, train_years)),
        },
        {
            "split": "holdout",
            "rows": len(holdout_raw),
            "n_parkings": holdout_raw["parking_id"].nunique(),
            "date_min": holdout_raw["rounded_hour"].min(),
            "date_max": holdout_raw["rounded_hour"].max(),
            "years": ", ".join(map(str, holdout_years)),
        },
    ]
)

display(scope_check_df)

,split,rows,n_parkings,date_min,date_max,years
0,train,129837,10,2020-01-01,2024-12-31 23:00:00,"2020, 2023, 2024"
1,holdout,87600,10,2025-01-01,2025-12-31 23:00:00,2025


## 2. EDA-grounded design summary for T/S/E

The feature choices below are constrained by Phase 2 findings:
- strong daily/weekly/monthly periodicity -> cyclical time features;
- stable spatial heterogeneity by parking and tier -> interpretable spatial encoding;
- external effects are heterogeneous and often non-linear -> conservative bins/thresholds plus selected interactions.

In [3]:
eda_design_summary_df = pd.DataFrame(
    [
        {
            "family": "T - time structure",
            "eda_motivation": "daily and weekly periodicity are strong and stable across Phase 2 temporal analyses",
            "implementation": "cyclical sin/cos encoding for hour, weekday, month + interpretable calendar flags",
        },
        {
            "family": "S - spatial identity",
            "eda_motivation": "clear parking-level and centrum-vs-vesten/rand heterogeneity",
            "implementation": "tier flag, log-capacity, capacity buckets, parking identity one-hot",
        },
        {
            "family": "E - external factors",
            "eda_motivation": "event and weather effects are context-dependent and often non-linear",
            "implementation": "precip bins, wind thresholds, sunshine transforms, event structure and selected interactions",
        },
    ]
)

display(eda_design_summary_df)

,family,eda_motivation,implementation
0,T - time structure,daily and weekly periodicity are strong and st...,"cyclical sin/cos encoding for hour, weekday, m..."
1,S - spatial identity,clear parking-level and centrum-vs-vesten/rand...,"tier flag, log-capacity, capacity buckets, par..."
2,E - external factors,event and weather effects are context-dependen...,"precip bins, wind thresholds, sunshine transfo..."


## A. T - Time structure

We build purely observation-based time features. No feature in this block uses target-derived information.

In [4]:
feature_registry_rows: list[dict[str, object]] = []


def register_features(
    feature_names: list[str],
    block: str,
    source_columns: str,
    track_allowed: str,
    requires_fit: bool,
    train_only_fit: bool,
    causal: bool,
    expected_role: str,
    notes: str,
) -> None:
    for name in feature_names:
        feature_registry_rows.append(
            {
                "feature_name": name,
                "block": block,
                "source_columns": source_columns,
                "track_allowed": track_allowed,
                "requires_fit": requires_fit,
                "train_only_fit": train_only_fit,
                "causal": causal,
                "expected_role": expected_role,
                "notes": notes,
            }
        )


SEASON_MAP = {
    12: "winter", 1: "winter", 2: "winter",
    3: "lente", 4: "lente", 5: "lente",
    6: "zomer", 7: "zomer", 8: "zomer",
    9: "herfst", 10: "herfst", 11: "herfst",
}


def add_time_features(df: pd.DataFrame, day_class_levels: list[str]) -> pd.DataFrame:
    out = df.copy()

    hour_sin, hour_cos = cyclical_encode(out["hour"], period=24)
    wd_sin, wd_cos = cyclical_encode(out["weekday_int"], period=7)
    month_sin, month_cos = cyclical_encode(out["month"], period=12)

    out["t_hour_sin"] = hour_sin
    out["t_hour_cos"] = hour_cos
    out["t_weekday_sin"] = wd_sin
    out["t_weekday_cos"] = wd_cos
    out["t_month_sin"] = month_sin
    out["t_month_cos"] = month_cos

    out["t_is_weekend"] = as_int_flag(out["weekday_int"].isin([5, 6]))
    out["t_is_national_holiday"] = as_int_flag(out["is_national_holiday"])
    out["t_is_other_holiday"] = as_int_flag(out["is_other_holiday"])
    out["t_is_any_holiday"] = as_int_flag(out["is_any_holiday"])
    out["t_is_school_vacation"] = as_int_flag(out["is_school_vacation"])
    out["t_is_2020_regime"] = pd.to_numeric(out["year"], errors="coerce").eq(2020).astype(np.int8)

    out["t_season"] = pd.to_numeric(out["month"], errors="coerce").map(SEASON_MAP).fillna("unknown")

    dayclass = np.select(
        [
            as_bool_flag(out["is_event_day"]),
            as_bool_flag(out["is_any_holiday"]),
            as_bool_flag(out["is_school_vacation"]),
            out["weekday_int"].isin([5, 6]),
        ],
        ["event_day", "holiday", "school_vacation", "weekend_regular"],
        default="weekday_regular",
    )
    out["t_day_class_5"] = pd.Categorical(dayclass, categories=day_class_levels)

    for level in day_class_levels:
        col = f"t_day_class_5__{slugify(level)}"
        out[col] = out["t_day_class_5"].eq(level).astype(np.int8)

    for season in ["winter", "lente", "zomer", "herfst"]:
        col = f"t_season__{season}"
        out[col] = out["t_season"].eq(season).astype(np.int8)

    return out


DAY_CLASS_LEVELS = ["weekday_regular", "weekend_regular", "school_vacation", "holiday", "event_day"]

train_t = add_time_features(train_raw, day_class_levels=DAY_CLASS_LEVELS)
holdout_t = add_time_features(holdout_raw, day_class_levels=DAY_CLASS_LEVELS)

register_features(
    feature_names=[
        "t_hour_sin", "t_hour_cos", "t_weekday_sin", "t_weekday_cos", "t_month_sin", "t_month_cos",
        "t_is_weekend", "t_is_national_holiday", "t_is_other_holiday", "t_is_any_holiday",
        "t_is_school_vacation", "t_is_2020_regime",
    ],
    block="T",
    source_columns="hour,weekday_int,month,is_national_holiday,is_other_holiday,is_any_holiday,is_school_vacation,year",
    track_allowed="policy|forecast",
    requires_fit=False,
    train_only_fit=False,
    causal=True,
    expected_role="cyclicity, calendar regime, and structural day-type shifts",
    notes="purely observation-based calendar and timestamp features",
)

register_features(
    feature_names=[f"t_day_class_5__{slugify(level)}" for level in DAY_CLASS_LEVELS],
    block="T",
    source_columns="is_event_day,is_any_holiday,is_school_vacation,weekday_int",
    track_allowed="policy|forecast",
    requires_fit=False,
    train_only_fit=False,
    causal=True,
    expected_role="compact day-type state feature",
    notes="derived categorical day class with fixed semantic categories",
)

register_features(
    feature_names=[f"t_season__{season}" for season in ["winter", "lente", "zomer", "herfst"]],
    block="T",
    source_columns="month",
    track_allowed="policy|forecast",
    requires_fit=False,
    train_only_fit=False,
    causal=True,
    expected_role="season controls for external interactions",
    notes="season derived from month mapping",
)

print("Time features added")
print("train_t shape:", train_t.shape)
print("holdout_t shape:", holdout_t.shape)

Time features added
train_t shape: (129837, 94)
holdout_t shape: (87600, 94)


### Ordinal vs cyclical representation (concept check)

Cyclical encoding is used as default because ordinal coding fails to represent wrap-around continuity (`23 -> 0`, `Dec -> Jan`) and can induce artificial linear distance.

In [5]:
concept_rows = [
    {
        "variable": "hour",
        "ordinal_distance(23,0)": abs(23 - 0),
        "cyclical_distance(23,0)": 1,
        "implication": "ordinal scale overstates boundary distance",
    },
    {
        "variable": "weekday_int",
        "ordinal_distance(6,0)": abs(6 - 0),
        "cyclical_distance(6,0)": 1,
        "implication": "Sunday-Monday continuity is preserved only with cyclical coding",
    },
    {
        "variable": "month",
        "ordinal_distance(12,1)": abs(12 - 1),
        "cyclical_distance(12,1)": 1,
        "implication": "year wrap-around is handled correctly",
    },
]

cyclical_concept_df = pd.DataFrame(concept_rows)
display(cyclical_concept_df)

,variable,"ordinal_distance(23,0)","cyclical_distance(23,0)",implication,"ordinal_distance(6,0)","cyclical_distance(6,0)","ordinal_distance(12,1)","cyclical_distance(12,1)"
0,hour,23.0,1.0,ordinal scale overstates boundary distance,NaN,NaN,NaN,NaN
1,weekday_int,NaN,NaN,Sunday-Monday continuity is preserved only wit...,6.0,1.0,NaN,NaN
2,month,NaN,NaN,year wrap-around is handled correctly,NaN,NaN,11.0,1.0


## B. S - Spatial identity

Spatial features are kept interpretable and compact.

Design decision:
- We keep leakage-safe parking one-hot encoding.
- We do **not** include target encoding in this shared immutable layer to avoid cross-fold leakage risk in later model selection.

In [6]:
PARKING_LEVELS = sorted(train_t["parking_id"].astype(str).unique().tolist())

cap_labels = ["small", "medium", "large"]
train_cap_bucket, cap_edges = qcut_with_fallback(train_t["total_capacity"], q=3, labels=cap_labels)
train_t["s_capacity_bucket"] = train_cap_bucket.astype(str)
holdout_t["s_capacity_bucket"] = cut_using_edges(holdout_t["total_capacity"], edges=cap_edges, labels=cap_labels).astype(str)


def add_spatial_features(df: pd.DataFrame, parking_levels: list[str]) -> pd.DataFrame:
    out = df.copy()

    out["s_tier_is_centrum"] = out["parking_location_category"].astype(str).str.lower().eq("centrum").astype(np.int8)
    out["s_tier_is_vesten"] = out["parking_location_category"].astype(str).str.lower().eq("vesten").astype(np.int8)
    out["s_tier_is_rand"] = out["parking_location_category"].astype(str).str.lower().eq("rand").astype(np.int8)

    cap_numeric = pd.to_numeric(out["total_capacity"], errors="coerce")
    out["s_log_total_capacity"] = np.log1p(cap_numeric)

    for level in ["small", "medium", "large"]:
        out[f"s_capacity_bucket__{level}"] = out["s_capacity_bucket"].eq(level).astype(np.int8)

    out["s_parking_id"] = pd.Categorical(out["parking_id"].astype(str), categories=parking_levels)
    for pid in parking_levels:
        col = f"s_parking_id__{slugify(pid)}"
        out[col] = out["s_parking_id"].eq(pid).astype(np.int8)

    return out


train_s = add_spatial_features(train_t, PARKING_LEVELS)
holdout_s = add_spatial_features(holdout_t, PARKING_LEVELS)

register_features(
    feature_names=["s_tier_is_centrum", "s_tier_is_vesten", "s_tier_is_rand", "s_log_total_capacity"],
    block="S",
    source_columns="parking_location_category,total_capacity",
    track_allowed="policy|forecast",
    requires_fit=False,
    train_only_fit=False,
    causal=True,
    expected_role="interpretable location and capacity context",
    notes="tier and capacity are stable metadata features",
)

register_features(
    feature_names=[f"s_capacity_bucket__{level}" for level in ["small", "medium", "large"]],
    block="S",
    source_columns="total_capacity",
    track_allowed="policy|forecast",
    requires_fit=True,
    train_only_fit=True,
    causal=True,
    expected_role="coarse capacity regime",
    notes="quantile bucket edges fit on train only",
)

register_features(
    feature_names=[f"s_parking_id__{slugify(pid)}" for pid in PARKING_LEVELS],
    block="S",
    source_columns="parking_id",
    track_allowed="policy|forecast",
    requires_fit=True,
    train_only_fit=True,
    causal=True,
    expected_role="parking-specific baseline heterogeneity",
    notes="one-hot categories fixed from train levels to preserve schema",
)

spatial_decision_df = pd.DataFrame(
    [
        {
            "decision": "parking identity encoding",
            "chosen_method": "train-frozen one-hot",
            "why": "low cardinality (10 parkings), high interpretability, no target leakage",
        },
        {
            "decision": "target encoding",
            "chosen_method": "not included in immutable core TSE layer",
            "why": "requires fold-aware refit to avoid leakage inside rolling CV",
        },
        {
            "decision": "behavioral clustering",
            "chosen_method": "not included in core",
            "why": "limited entity count and stability risk; can be tested in robustness extension",
        },
    ]
)

display(spatial_decision_df)
print("Spatial features added")
print("train_s shape:", train_s.shape)
print("holdout_s shape:", holdout_s.shape)

,decision,chosen_method,why
0,parking identity encoding,train-frozen one-hot,"low cardinality (10 parkings), high interpreta..."
1,target encoding,not included in immutable core TSE layer,requires fold-aware refit to avoid leakage ins...
2,behavioral clustering,not included in core,limited entity count and stability risk; can b...


Spatial features added
train_s shape: (129837, 113)
holdout_s shape: (87600, 113)


## C. E - External factors

External block design follows Phase 2 guidance: non-linearity where needed, no raw free-text predictors, and controlled interaction count.

In [7]:
# Fit train-only parameters for external transforms
precip_positive = pd.to_numeric(train_s["precip_mm"], errors="coerce")
precip_positive = precip_positive[precip_positive > 0]
if len(precip_positive) > 0:
    precip_q50 = float(precip_positive.quantile(0.50))
    precip_q90 = float(precip_positive.quantile(0.90))
else:
    precip_q50, precip_q90 = 0.1, 1.0

wind_speed_high_thr = float(pd.to_numeric(train_s["wind_speed_ms"], errors="coerce").quantile(0.90))
wind_gust_high_thr = float(pd.to_numeric(train_s["wind_gusts_ms"], errors="coerce").quantile(0.90))

n_concurrent_cap = int(pd.to_numeric(train_s["n_concurrent_events"], errors="coerce").quantile(0.95))
if n_concurrent_cap < 1:
    n_concurrent_cap = 1


def add_external_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    temp = pd.to_numeric(out["temp_c"], errors="coerce")
    precip = pd.to_numeric(out["precip_mm"], errors="coerce")
    wind = pd.to_numeric(out["wind_speed_ms"], errors="coerce")
    gust = pd.to_numeric(out["wind_gusts_ms"], errors="coerce")
    sun_duration = pd.to_numeric(out["sun_duration_min"], errors="coerce")
    sun_intensity = pd.to_numeric(out["sun_intensity_wm2"], errors="coerce")

    # Weather transforms
    out["e_temp_c"] = temp
    out["e_temp_c_sq"] = np.square(temp)

    out["e_precip_mm"] = precip
    out["e_precip_bin"] = np.select(
        [
            precip.eq(0),
            (precip > 0) & (precip <= precip_q50),
            (precip > precip_q50) & (precip <= precip_q90),
            precip > precip_q90,
        ],
        ["dry", "light", "moderate", "heavy"],
        default="unknown",
    )
    for lvl in ["dry", "light", "moderate", "heavy"]:
        out[f"e_precip_bin__{lvl}"] = pd.Series(out["e_precip_bin"]).eq(lvl).astype(np.int8)

    out["e_wind_speed_ms"] = wind
    out["e_wind_gusts_ms"] = gust
    out["e_is_high_wind_speed"] = wind.ge(wind_speed_high_thr).fillna(False).astype(np.int8)
    out["e_is_high_wind_gust"] = gust.ge(wind_gust_high_thr).fillna(False).astype(np.int8)

    out["e_sun_duration_ratio"] = (sun_duration / 60.0).clip(lower=0, upper=1)
    out["e_sun_intensity_log1p"] = np.log1p(sun_intensity.clip(lower=0))

    # Event and calendar-external structure
    out["e_is_event_day"] = as_int_flag(out["is_event_day"])
    for col in [
        "is_football_day", "is_sport_day", "is_festival_day", "is_procession_day",
        "is_kermis_day", "is_markt_day", "is_carnival_day", "is_other_day",
    ]:
        out[f"e_{col}"] = as_int_flag(out[col])

    out["e_event_scale_max"] = pd.to_numeric(out["event_scale_max"], errors="coerce").fillna(0).astype(int)
    out["e_event_scale_is_large"] = out["e_event_scale_max"].ge(2).astype(np.int8)

    out["e_n_concurrent_events"] = pd.to_numeric(out["n_concurrent_events"], errors="coerce").fillna(0)
    out["e_n_concurrent_events_capped"] = out["e_n_concurrent_events"].clip(upper=n_concurrent_cap)

    kickoff = pd.to_numeric(out["football_kickoff_hour"], errors="coerce")
    out["e_has_football_kickoff"] = kickoff.notna().astype(np.int8)
    kickoff_sin, kickoff_cos = cyclical_encode(kickoff.fillna(0), period=24)
    out["e_football_kickoff_hour_sin"] = kickoff_sin
    out["e_football_kickoff_hour_cos"] = kickoff_cos

    return out


train_e = add_external_features(train_s)
holdout_e = add_external_features(holdout_s)

register_features(
    feature_names=[
        "e_temp_c", "e_temp_c_sq", "e_precip_mm", "e_wind_speed_ms", "e_wind_gusts_ms",
        "e_is_high_wind_speed", "e_is_high_wind_gust", "e_sun_duration_ratio", "e_sun_intensity_log1p",
    ],
    block="E",
    source_columns="temp_c,precip_mm,wind_speed_ms,wind_gusts_ms,sun_duration_min,sun_intensity_wm2",
    track_allowed="policy|forecast",
    requires_fit=True,
    train_only_fit=True,
    causal=True,
    expected_role="non-linear and threshold-aware weather representation",
    notes="precip and wind thresholds are fit on train only",
)

register_features(
    feature_names=[f"e_precip_bin__{lvl}" for lvl in ["dry", "light", "moderate", "heavy"]],
    block="E",
    source_columns="precip_mm",
    track_allowed="policy|forecast",
    requires_fit=True,
    train_only_fit=True,
    causal=True,
    expected_role="non-linear precipitation effects",
    notes="dry/light/moderate/heavy bins from train quantiles",
)

register_features(
    feature_names=[
        "e_is_event_day", "e_is_football_day", "e_is_sport_day", "e_is_festival_day", "e_is_procession_day",
        "e_is_kermis_day", "e_is_markt_day", "e_is_carnival_day", "e_is_other_day",
        "e_event_scale_max", "e_event_scale_is_large", "e_n_concurrent_events", "e_n_concurrent_events_capped",
        "e_has_football_kickoff", "e_football_kickoff_hour_sin", "e_football_kickoff_hour_cos",
    ],
    block="E",
    source_columns="is_event_day,event_type_flags,event_scale_max,n_concurrent_events,football_kickoff_hour",
    track_allowed="policy|forecast",
    requires_fit=True,
    train_only_fit=True,
    causal=True,
    expected_role="event intensity and event-type heterogeneity",
    notes="no raw text event fields are used",
)

fit_params_external = {
    "precip_positive_q50": precip_q50,
    "precip_positive_q90": precip_q90,
    "wind_speed_high_thr_q90": wind_speed_high_thr,
    "wind_gust_high_thr_q90": wind_gust_high_thr,
    "n_concurrent_events_cap_q95": n_concurrent_cap,
}

print("External features added")
print("train_e shape:", train_e.shape)
print("holdout_e shape:", holdout_e.shape)
print("Fitted external params:")
print(fit_params_external)

External features added
train_e shape: (129837, 143)
holdout_e shape: (87600, 143)
Fitted external params:
{'precip_positive_q50': 0.28, 'precip_positive_q90': 1.77, 'wind_speed_high_thr_q90': 5.5, 'wind_gust_high_thr_q90': 10.82, 'n_concurrent_events_cap_q95': 2}


### Interactions (controlled, EDA-motivated)

Only interaction families with clear EDA rationale are included:
- `event x tier`
- `holiday x tier`
- `vacation x tier`
- selected `weather x season`

In [8]:
def add_external_interactions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["e_int_event_x_tier_centrum"] = out["e_is_event_day"] * out["s_tier_is_centrum"]
    out["e_int_holiday_x_tier_centrum"] = out["t_is_any_holiday"] * out["s_tier_is_centrum"]
    out["e_int_vacation_x_tier_centrum"] = out["t_is_school_vacation"] * out["s_tier_is_centrum"]

    for season in ["winter", "lente", "zomer", "herfst"]:
        season_col = f"t_season__{season}"
        out[f"e_int_temp_x_season__{season}"] = out["e_temp_c"] * out[season_col]
        out[f"e_int_precip_x_season__{season}"] = out["e_precip_mm"] * out[season_col]
        out[f"e_int_wind_x_season__{season}"] = out["e_wind_speed_ms"] * out[season_col]

    return out


train_int = add_external_interactions(train_e)
holdout_int = add_external_interactions(holdout_e)

register_features(
    feature_names=[
        "e_int_event_x_tier_centrum",
        "e_int_holiday_x_tier_centrum",
        "e_int_vacation_x_tier_centrum",
    ],
    block="E",
    source_columns="e_is_event_day,t_is_any_holiday,t_is_school_vacation,s_tier_is_centrum",
    track_allowed="policy|forecast",
    requires_fit=False,
    train_only_fit=False,
    causal=True,
    expected_role="tier-specific external sensitivity",
    notes="interaction count intentionally limited",
)

weather_season_interaction_cols = []
for season in ["winter", "lente", "zomer", "herfst"]:
    weather_season_interaction_cols.extend(
        [
            f"e_int_temp_x_season__{season}",
            f"e_int_precip_x_season__{season}",
            f"e_int_wind_x_season__{season}",
        ]
    )

register_features(
    feature_names=weather_season_interaction_cols,
    block="E",
    source_columns="e_temp_c,e_precip_mm,e_wind_speed_ms,t_season one-hot",
    track_allowed="policy|forecast",
    requires_fit=False,
    train_only_fit=False,
    causal=True,
    expected_role="season-modulated weather effects",
    notes="kept compact to avoid interaction explosion",
)

print("Interaction features added")

Interaction features added


### Event cascade features without leakage

Critical design rationale:
- `hours_to_event` uses only event-calendar information (not occupancy), so no target leakage.
- This is valid only under an ex-ante event-availability assumption (calendar known before prediction time).
- Features are clipped to a finite window and use a sentinel value when no relevant event is nearby.

In this notebook:
- clip window = `72` hours
- sentinel value = `73` (means: no event in the relevant 72h window)

In [9]:
combined_for_cascade = pd.concat(
    [
        train_int[["rounded_hour", "is_event_day"]],
        holdout_int[["rounded_hour", "is_event_day"]],
    ],
    axis=0,
    ignore_index=True,
)

cascade_map = compute_event_cascade_features(
    timestamp_series=combined_for_cascade["rounded_hour"],
    event_series=combined_for_cascade["is_event_day"],
    clip_hours=72,
)

train_cascade = train_int.merge(cascade_map, on="rounded_hour", how="left")
holdout_cascade = holdout_int.merge(cascade_map, on="rounded_hour", how="left")

for col in [
    "e_hours_to_event_clip",
    "e_hours_since_event_clip",
    "e_event_proximity_clip",
    "e_event_window_pre_24h",
    "e_event_window_post_24h",
]:
    train_cascade[col] = pd.to_numeric(train_cascade[col], errors="coerce").fillna(73).astype(int)
    holdout_cascade[col] = pd.to_numeric(holdout_cascade[col], errors="coerce").fillna(73).astype(int)

register_features(
    feature_names=[
        "e_hours_to_event_clip",
        "e_hours_since_event_clip",
        "e_event_proximity_clip",
        "e_event_window_pre_24h",
        "e_event_window_post_24h",
    ],
    block="E",
    source_columns="rounded_hour,is_event_day (event calendar timeline)",
    track_allowed="policy|forecast",
    requires_fit=False,
    train_only_fit=False,
    causal=True,
    expected_role="event proximity and pre/post event dynamics",
    notes="clip=72h, sentinel=73; no free-text event source",
)

cascade_check_df = pd.DataFrame(
    [
        {
            "split": "train",
            "hours_to_event_min": train_cascade["e_hours_to_event_clip"].min(),
            "hours_to_event_max": train_cascade["e_hours_to_event_clip"].max(),
            "hours_since_event_min": train_cascade["e_hours_since_event_clip"].min(),
            "hours_since_event_max": train_cascade["e_hours_since_event_clip"].max(),
            "sentinel_rate_hours_to_event_pct": float((train_cascade["e_hours_to_event_clip"] == 73).mean() * 100),
            "sentinel_rate_hours_since_event_pct": float((train_cascade["e_hours_since_event_clip"] == 73).mean() * 100),
        },
        {
            "split": "holdout",
            "hours_to_event_min": holdout_cascade["e_hours_to_event_clip"].min(),
            "hours_to_event_max": holdout_cascade["e_hours_to_event_clip"].max(),
            "hours_since_event_min": holdout_cascade["e_hours_since_event_clip"].min(),
            "hours_since_event_max": holdout_cascade["e_hours_since_event_clip"].max(),
            "sentinel_rate_hours_to_event_pct": float((holdout_cascade["e_hours_to_event_clip"] == 73).mean() * 100),
            "sentinel_rate_hours_since_event_pct": float((holdout_cascade["e_hours_since_event_clip"] == 73).mean() * 100),
        },
    ]
)

display(cascade_check_df.round(3))

,split,hours_to_event_min,hours_to_event_max,hours_since_event_min,hours_since_event_max,sentinel_rate_hours_to_event_pct,sentinel_rate_hours_since_event_pct
0,train,0,72,0,73,0.000,11.889
1,holdout,0,73,0,72,1.096,0.000


## D. Validation and registry update

Systematic checks:
- missingness per feature
- constant columns
- cardinality
- train/holdout schema equivalence
- leakage-risk sanity checks

In [10]:
train_feat = train_cascade.copy()
holdout_feat = holdout_cascade.copy()

# Feature selection: keep keys, target, and engineered T/S/E features only
helper_categorical_cols = {
    "t_season",
    "t_day_class_5",
    "s_capacity_bucket",
    "s_parking_id",
    "e_precip_bin",
}

feature_cols = sorted(
    [
        c
        for c in train_feat.columns
        if (c.startswith("t_") or c.startswith("s_") or c.startswith("e_")) and c not in helper_categorical_cols
    ]
)

final_cols = KEY_COLS + feature_cols
train_final = train_feat[final_cols].copy()
holdout_final = holdout_feat[final_cols].copy()

# Validation 1: missingness
missing_train = train_final[feature_cols].isna().mean().rename("missing_rate_train")
missing_holdout = holdout_final[feature_cols].isna().mean().rename("missing_rate_holdout")
missing_df = pd.concat([missing_train, missing_holdout], axis=1).reset_index().rename(columns={"index": "feature_name"})

# Validation 2: constant columns
constant_train = [c for c in feature_cols if train_final[c].nunique(dropna=False) <= 1]
constant_holdout = [c for c in feature_cols if holdout_final[c].nunique(dropna=False) <= 1]

constant_df = pd.DataFrame(
    {
        "constant_in_train": [", ".join(constant_train) if constant_train else "none"],
        "constant_in_holdout": [", ".join(constant_holdout) if constant_holdout else "none"],
    }
)

# Validation 3: cardinality
cardinality_df = pd.DataFrame(
    {
        "feature_name": feature_cols,
        "n_unique_train": [train_final[c].nunique(dropna=False) for c in feature_cols],
        "n_unique_holdout": [holdout_final[c].nunique(dropna=False) for c in feature_cols],
    }
).sort_values("n_unique_train", ascending=False)

# Validation 4: schema equivalence
schema_train = train_final.dtypes.astype(str).rename("dtype_train")
schema_holdout = holdout_final.dtypes.astype(str).rename("dtype_holdout")
schema_check_df = pd.concat([schema_train, schema_holdout], axis=1).reset_index().rename(columns={"index": "column"})
schema_check_df["dtype_match"] = schema_check_df["dtype_train"].eq(schema_check_df["dtype_holdout"])

# Validation 5: leakage-risk checks
forbidden_raw_cols = {
    "event_names_combined",
    "number_of_occupied_spaces",
    "occupancy_rate",
    "low_data_coverage",
    "system_blackout",
    "partial_year",
}

feature_registry_df = pd.DataFrame(feature_registry_rows).drop_duplicates(subset=["feature_name"]).sort_values(["block", "feature_name"]) 

registry_sources_joined = feature_registry_df["source_columns"].astype(str).str.cat(sep=",")
contains_forbidden_source = any(token in registry_sources_joined for token in forbidden_raw_cols)
contains_lag_feature = any(("lag" in c.lower()) or ("rolling" in c.lower()) for c in feature_cols)

leakage_checks_df = pd.DataFrame(
    [
        {
            "check": "forbidden raw text field used",
            "result": "PASS" if "event_names_combined" not in registry_sources_joined else "FAIL",
            "detail": "event_names_combined is excluded from source_columns",
        },
        {
            "check": "target proxy used as predictor",
            "result": "PASS" if not any("number_of_occupied_spaces" in s for s in feature_registry_df["source_columns"].astype(str)) else "FAIL",
            "detail": "number_of_occupied_spaces excluded from engineered features",
        },
        {
            "check": "occupancy lag or rolling present in TSE layer",
            "result": "PASS" if not contains_lag_feature else "FAIL",
            "detail": "no lag/rolling features are built in fe_01",
        },
        {
            "check": "schema train vs holdout identical",
            "result": "PASS" if schema_check_df["dtype_match"].all() else "FAIL",
            "detail": "all columns and dtypes aligned",
        },
    ]
)

print("Missingness summary (top 20 by train missing rate)")
display(missing_df.sort_values("missing_rate_train", ascending=False).head(20))

print("Constant-column check")
display(constant_df)

print("Cardinality summary (top 25)")
display(cardinality_df.head(25))

print("Schema equivalence check")
display(schema_check_df)

print("Leakage-risk checks")
display(leakage_checks_df)

assert not contains_lag_feature, "Lag-like feature detected in TSE layer"
assert schema_check_df["dtype_match"].all(), "Train/holdout schema mismatch"
assert "event_names_combined" not in registry_sources_joined, "Raw event text leakage risk detected"

Missingness summary (top 20 by train missing rate)


,feature_name,missing_rate_train,missing_rate_holdout
0,e_event_proximity_clip,0.0,0.0
55,s_parking_id__p_hoogstraat,0.0,0.0
63,s_tier_is_centrum,0.0,0.0
62,s_parking_id__p_veemarkt,0.0,0.0
61,s_parking_id__p_tinel,0.0,0.0
60,s_parking_id__p_maarten,0.0,0.0
59,s_parking_id__p_lamot,0.0,0.0
58,s_parking_id__p_komet,0.0,0.0
57,s_parking_id__p_keerdok,0.0,0.0
56,s_parking_id__p_kathedraal,0.0,0.0


Constant-column check


,constant_in_train,constant_in_holdout
0,none,t_is_2020_regime


Cardinality summary (top 25)


,feature_name,n_unique_train,n_unique_holdout
44,e_sun_intensity_log1p,9442,3727
45,e_temp_c,3449,2808
46,e_temp_c_sq,3163,2590
16,e_int_temp_x_season__herfst,2154,1243
17,e_int_temp_x_season__lente,2065,1381
19,e_int_temp_x_season__zomer,1976,1249
18,e_int_temp_x_season__winter,1661,1229
48,e_wind_speed_ms,1035,725
23,e_int_wind_x_season__winter,901,623
22,e_int_wind_x_season__lente,793,548


Schema equivalence check


,column,dtype_train,dtype_holdout,dtype_match
0,parking_id,str,str,True
1,rounded_hour,datetime64[ns],datetime64[ns],True
2,year,Int64,Int64,True
3,occupancy_rate,float64,float64,True
4,operational_split,str,str,True
5,e_event_proximity_clip,int64,int64,True
6,e_event_scale_is_large,int8,int8,True
7,e_event_scale_max,int64,int64,True
8,e_event_window_post_24h,int64,int64,True
9,e_event_window_pre_24h,int64,int64,True


Leakage-risk checks


,check,result,detail
0,forbidden raw text field used,PASS,event_names_combined is excluded from source_c...
1,target proxy used as predictor,PASS,number_of_occupied_spaces excluded from engine...
2,occupancy lag or rolling present in TSE layer,PASS,no lag/rolling features are built in fe_01
3,schema train vs holdout identical,PASS,all columns and dtypes aligned


In [11]:
feature_registry_df = feature_registry_df.reset_index(drop=True)

# Data dictionary is a compact view on top of registry
feature_dictionary_df = feature_registry_df.rename(
    columns={
        "feature_name": "column_name",
        "expected_role": "definition",
    }
)[
    [
        "column_name",
        "block",
        "definition",
        "source_columns",
        "track_allowed",
        "notes",
    ]
]

registry_block_summary_df = (
    feature_registry_df.groupby(["block", "track_allowed"], as_index=False)
    .size()
    .rename(columns={"size": "n_features"})
    .sort_values(["block", "track_allowed"])
)

print("Feature registry block summary")
display(registry_block_summary_df)

print("Feature registry (first 30)")
display(feature_registry_df.head(30))

Feature registry block summary


,block,track_allowed,n_features
0,E,policy|forecast,49
1,S,policy|forecast,17
2,T,policy|forecast,21


Feature registry (first 30)


,feature_name,block,source_columns,track_allowed,requires_fit,train_only_fit,causal,expected_role,notes
0,e_event_proximity_clip,E,"rounded_hour,is_event_day (event calendar time...",policy|forecast,False,False,True,event proximity and pre/post event dynamics,"clip=72h, sentinel=73; no free-text event source"
1,e_event_scale_is_large,E,"is_event_day,event_type_flags,event_scale_max,...",policy|forecast,True,True,True,event intensity and event-type heterogeneity,no raw text event fields are used
2,e_event_scale_max,E,"is_event_day,event_type_flags,event_scale_max,...",policy|forecast,True,True,True,event intensity and event-type heterogeneity,no raw text event fields are used
3,e_event_window_post_24h,E,"rounded_hour,is_event_day (event calendar time...",policy|forecast,False,False,True,event proximity and pre/post event dynamics,"clip=72h, sentinel=73; no free-text event source"
4,e_event_window_pre_24h,E,"rounded_hour,is_event_day (event calendar time...",policy|forecast,False,False,True,event proximity and pre/post event dynamics,"clip=72h, sentinel=73; no free-text event source"
5,e_football_kickoff_hour_cos,E,"is_event_day,event_type_flags,event_scale_max,...",policy|forecast,True,True,True,event intensity and event-type heterogeneity,no raw text event fields are used
6,e_football_kickoff_hour_sin,E,"is_event_day,event_type_flags,event_scale_max,...",policy|forecast,True,True,True,event intensity and event-type heterogeneity,no raw text event fields are used
7,e_has_football_kickoff,E,"is_event_day,event_type_flags,event_scale_max,...",policy|forecast,True,True,True,event intensity and event-type heterogeneity,no raw text event fields are used
8,e_hours_since_event_clip,E,"rounded_hour,is_event_day (event calendar time...",policy|forecast,False,False,True,event proximity and pre/post event dynamics,"clip=72h, sentinel=73; no free-text event source"
9,e_hours_to_event_clip,E,"rounded_hour,is_event_day (event calendar time...",policy|forecast,False,False,True,event proximity and pre/post event dynamics,"clip=72h, sentinel=73; no free-text event source"


## E. Export shared TSE layer and artifacts

This export is shared by both tracks:
- policy will directly consume `T+S+E`
- forecast will consume this as base and add `L` in `fe_02`

In [12]:
train_out_path = OUTPUT_DIR / "tse_core_train_2020_2023_2024.parquet"
holdout_out_path = OUTPUT_DIR / "tse_core_holdout_2025.parquet"
registry_out_path = OUTPUT_DIR / "feature_registry_tse.csv"
dict_out_path = OUTPUT_DIR / "data_dictionary_tse.csv"
fit_params_out_path = OUTPUT_DIR / "tse_fit_params.json"
schema_out_path = OUTPUT_DIR / "tse_schema.json"
manifest_out_path = OUTPUT_DIR / "tse_export_manifest.csv"

train_final.to_parquet(train_out_path, index=False)
holdout_final.to_parquet(holdout_out_path, index=False)
feature_registry_df.to_csv(registry_out_path, index=False)
feature_dictionary_df.to_csv(dict_out_path, index=False)

fit_params_payload = {
    "capacity_bin_edges": [float(x) for x in np.array(cap_edges).tolist()],
    "parking_levels": PARKING_LEVELS,
    "day_class_levels": DAY_CLASS_LEVELS,
    "external_fit_params": fit_params_external,
    "event_cascade": {"clip_hours": 72, "sentinel": 73},
    "note": "All fit-dependent parameters estimated on train split only.",
}
fit_params_out_path.write_text(json.dumps(fit_params_payload, indent=2))

schema_payload = {
    "columns": train_final.columns.tolist(),
    "dtypes": {col: str(dtype) for col, dtype in train_final.dtypes.items()},
}
schema_out_path.write_text(json.dumps(schema_payload, indent=2))

manifest_df = pd.DataFrame(
    [
        {"artifact": "train_tse_parquet", "path": str(train_out_path), "n_rows": len(train_final), "n_cols": train_final.shape[1]},
        {"artifact": "holdout_tse_parquet", "path": str(holdout_out_path), "n_rows": len(holdout_final), "n_cols": holdout_final.shape[1]},
        {"artifact": "feature_registry_csv", "path": str(registry_out_path), "n_rows": len(feature_registry_df), "n_cols": feature_registry_df.shape[1]},
        {"artifact": "data_dictionary_csv", "path": str(dict_out_path), "n_rows": len(feature_dictionary_df), "n_cols": feature_dictionary_df.shape[1]},
        {"artifact": "fit_params_json", "path": str(fit_params_out_path), "n_rows": 1, "n_cols": 1},
        {"artifact": "schema_json", "path": str(schema_out_path), "n_rows": 1, "n_cols": 1},
    ]
)
manifest_df.to_csv(manifest_out_path, index=False)

print("TSE exports completed")
display(manifest_df)

TSE exports completed


,artifact,path,n_rows,n_cols
0,train_tse_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,129837,92
1,holdout_tse_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,87600,92
2,feature_registry_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,87,9
3,data_dictionary_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,87,6
4,fit_params_json,/Users/emilevandevoorde/Documents/mp_mechelen_...,1,1
5,schema_json,/Users/emilevandevoorde/Documents/mp_mechelen_...,1,1


## Academic close-out

### Which feature families were built?
The notebook produced a shared core layer with:
- `T`: cyclical temporal structure, calendar flags, regime indicator, and compact day-class encoding.
- `S`: interpretable spatial identity through tier indicators, capacity context, and train-frozen parking one-hot encoding.
- `E`: non-linear weather transforms, event-type structure, event intensity measures, selected interactions, and leakage-aware event-cascade timing features.

### Which interactions were retained (or intentionally excluded)?
Retained interactions:
- `event x tier`
- `holiday x tier`
- `vacation x tier`
- compact `weather x season` interactions.

Intentionally excluded in this core layer:
- high-order interaction combinations that would create feature explosion.
- behavioral cluster interactions (stability not yet robust at this stage).

### Which risks remain?
Residual risks remain in:
- ex-ante event calendar quality (timing uncertainty can weaken cascade precision).
- possible temporal drift of external effects from train years to 2025 holdout.
- threshold sensitivity (precipitation and wind cut-offs) despite train-only fitting discipline.

### Next step toward forecast-only L features
`fe_02_build_forecast_lag_features_L.ipynb` will add forecast-only autoregressive features (`L`) on top of this immutable TSE base, with strict causal lag computation and additional leakage checks.

## What was built
A reusable, leakage-safe, shared `T+S+E` feature layer for train and holdout, plus registry, dictionary, schema, and train-only fit artifacts.

## Leakage and validity checks
The notebook enforces split lock, train-only fitting for all fitted transforms, exclusion of raw text predictors, no occupancy lags, and schema equivalence across train and holdout.

## Implications for next notebook
The forecast track can now extend this base with `L` in `fe_02` without changing the already locked core TSE design.